In [1]:
import pandas as pd, sklearn, xgboost
from pathlib import Path

BRUTOS = Path("../dados/brutos")
PROC   = Path("../dados/processados")

print(pd.__version__, sklearn.__version__, xgboost.__version__)

3.0.5 1.9.1 3.4.1


In [2]:
import pandas as pd
from pathlib import Path

BRUTOS = Path("../dados/brutos")

for arq in sorted(BRUTOS.glob("*.csv")):
    df = pd.read_csv(arq)
    print(f"\n{arq.name}  →  {df.shape[0]:,} x {df.shape[1]}")
    print(df.columns[:6].tolist(), "...")


PhiUSIIL_Phishing_URL_Dataset.csv  →  235,795 x 56
['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP'] ...

dataset_B_05_2020.csv  →  11,430 x 89
['url', 'length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens'] ...


In [3]:
import pandas as pd
from pathlib import Path

BRUTOS = Path("../dados/brutos")

# confere o que tem na pasta
print([a.name for a in sorted(BRUTOS.glob("*.csv"))])

['PhiUSIIL_Phishing_URL_Dataset.csv', 'dataset_B_05_2020.csv']


In [4]:
hann = pd.read_csv(BRUTOS / "dataset_B_05_2020.csv")
print(hann.shape)

for col in hann.select_dtypes(include="object").columns:
    amostra = hann[col].dropna().astype(str).head(3).tolist()
    if any(v.startswith("http") for v in amostra):
        print(f"\n→ {col}")
        for v in amostra:
            print(f"   {v[:80]}")

(11430, 89)

→ url
   http://www.crestonwood.com/router.php
   http://shadetreetechnology.com/V4/validation/a111aedc8ae390eabcfa130e041a10a4
   https://support-appleld.com.secureupdate.duilawyeryork.com/ap/89e6a3b4b063b8d/?c


/tmp/ipykernel_61466/1703931297.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in hann.select_dtypes(include="object").columns:


In [5]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from extracao_lexica import extrair_dataframe

In [6]:
phiusiil = pd.read_csv(BRUTOS / "PhiUSIIL_Phishing_URL_Dataset.csv")

X_treino = extrair_dataframe(phiusiil["URL"])
X_teste  = extrair_dataframe(hann["url"])

print(X_treino.shape, X_teste.shape)
assert list(X_treino.columns) == list(X_teste.columns)

(235795, 37) (11430, 37)


In [7]:
print(phiusiil["label"].value_counts())
print(hann["status"].value_counts())

label
1    134850
0    100945
Name: count, dtype: int64
status
legitimate    5715
phishing      5715
Name: count, dtype: int64


In [8]:
y_treino = (phiusiil["label"] == 0).astype(int)   # phishing = 1
y_teste  = (hann["status"] == "phishing").astype(int)

print("treino:", y_treino.value_counts(normalize=True).round(3).to_dict())
print("teste :", y_teste.value_counts(normalize=True).round(3).to_dict())

treino: {0: 0.572, 1: 0.428}
teste : {0: 0.5, 1: 0.5}


In [9]:
print("duplicatas no PhiUSIIL:", phiusiil["URL"].duplicated().sum())
print("duplicatas no Hannousse:", hann["url"].duplicated().sum())

sobreposicao = set(phiusiil["URL"]) & set(hann["url"])
print("URLs em comum entre os dois datasets:", len(sobreposicao))

duplicatas no PhiUSIIL: 425
duplicatas no Hannousse: 1


URLs em comum entre os dois datasets: 44


In [10]:
dup = phiusiil[phiusiil["URL"].duplicated(keep=False)]
conflitos = dup.groupby("URL")["label"].nunique()
print("URLs com rótulos contraditórios:", (conflitos > 1).sum())

URLs com rótulos contraditórios: 0


In [11]:
# 1. PhiUSIIL: remover duplicatas
mask_tr = ~phiusiil["URL"].duplicated(keep="first")
phiusiil_limpo = phiusiil[mask_tr]

X_treino = X_treino[mask_tr.values]
y_treino = y_treino[mask_tr.values]

# 2. Hannousse: remover duplicatas e sobreposição com o treino
mask_te = (
    ~hann["url"].duplicated(keep="first")
    & ~hann["url"].isin(set(phiusiil_limpo["URL"]))
)
hann_limpo = hann[mask_te]

X_teste = X_teste[mask_te.values]
y_teste = y_teste[mask_te.values]

print("treino:", X_treino.shape, "| teste independente:", X_teste.shape)
print("phishing no treino:", y_treino.mean().round(3))
print("phishing no teste :", y_teste.mean().round(3))

treino: (235370, 37) | teste independente: (11385, 37)
phishing no treino: 0.427
phishing no teste : 0.498


In [12]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_treino, y_treino,
    test_size=0.2,
    stratify=y_treino,
    random_state=42,
)

urls_tr  = set(phiusiil_limpo.loc[X_tr.index, "URL"])
urls_val = set(phiusiil_limpo.loc[X_val.index, "URL"])
print(X_tr.shape, X_val.shape)
print("vazamento treino/validação:", len(urls_tr & urls_val))

(188296, 37) (47074, 37)
vazamento treino/validação: 0


In [13]:
for nome, obj in [("X_tr", X_tr), ("X_val", X_val), ("X_teste", X_teste)]:
    obj.to_csv(PROC / f"{nome}.csv", index=True)

for nome, obj in [("y_tr", y_tr), ("y_val", y_val), ("y_teste", y_teste)]:
    obj.to_frame("y").to_csv(PROC / f"{nome}.csv", index=True)